In [ ]:
import os
print(os.getcwd())

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU! Go to Runtime > Change runtime type > T4 GPU'
print('GPU   :', torch.cuda.get_device_name(0))
print('VRAM  :', round(torch.cuda.get_device_properties(0).total_memory/1e9,1), 'GB')
print('PyTorch:', torch.__version__)



In [ ]:
import os
if not os.path.exists('/content/LoGoHTR'):
    !git clone https://github.com/shreemitra/LoGoHTR.git
else:
    print('Already cloned, skipping.')
%cd /content/LoGoHTR
!ls

In [ ]:
!pip install -q einops editdistance 'jsonargparse[signatures]' --no-deps
!pip install -q pytorch-lightning --quiet

# Add repo to path instead of broken pip install -e .
import sys
sys.path.insert(0, '/content/LoGoHTR')
sys.path.insert(0, '/content/LoGoHTR/comer')
print('Done!')

In [ ]:
# Check what's inside the cloned repo first
import os
for item in os.listdir('.'):
    print(item)

In [ ]:

import shutil

total, used, free = shutil.disk_usage('/content')
print(f"Total: {total/1e9:.1f} GB")
print(f"Used: {used/1e9:.1f} GB")
print(f"Free: {free/1e9:.1f} GB")

# How many writers can we fit?
# Average writer folder is ~100MB
safe_space = (free/1e9) - 5  # keep 5GB buffer
writers_possible = int(safe_space / 0.1)
print(f"\nSafe usable space: {safe_space:.1f} GB")
print(f"Estimated writers we can extract: {writers_possible}")

In [ ]:
!pip install -q lightning einops editdistance 'jsonargparse[signatures]>=4.27.7' opencv-python matplotlib typer beautifulsoup4 lxml

In [ ]:
!wget -q "https://huggingface.co/Shree10/LoGo-HTR/resolve/main/DenseNet_backbone_step290000.pth" \
-O "/kaggle/working/LoGoHTR/comer/model/BarlowTwins_HTR_backbone_step290000.pth"
print("Backbone downloaded!")

In [ ]:
import os

# Fix pytorch_lightning imports
for root, dirs, files in os.walk('/kaggle/working/LoGoHTR/comer'):
    for file in files:
        if file.endswith('.py'):
            fpath = os.path.join(root, file)
            with open(fpath, 'r') as f:
                content = f.read()
            if 'pytorch_lightning' in content:
                content = content.replace('import pytorch_lightning', 'import lightning.pytorch')
                content = content.replace('from pytorch_lightning', 'from lightning.pytorch')
                with open(fpath, 'w') as f:
                    f.write(content)
print("Imports fixed!")

# Fix vocab
with open('/kaggle/working/LoGoHTR/comer/datamodule/vocab.py', 'r') as f:
    content = f.read()
content = content.replace('IIIT_Dict.txt', 'SSL_HWD_Dict.txt')
with open('/kaggle/working/LoGoHTR/comer/datamodule/vocab.py', 'w') as f:
    f.write(content)
print("Vocab fixed!")

# Fix encoder path
with open('/kaggle/working/LoGoHTR/comer/model/encoder.py', 'r') as f:
    content = f.read()
content = content.replace(
    '/home2/shree.mitra23m/CoMER/comer/model/BarlowTwins_HTR_backbone_step290000.pth',
    '/kaggle/working/LoGoHTR/comer/model/BarlowTwins_HTR_backbone_step290000.pth'
)
with open('/kaggle/working/LoGoHTR/comer/model/encoder.py', 'w') as f:
    f.write(content)
print("Encoder fixed!")

In [ ]:
new_train = """
import sys
sys.path.insert(0, '/kaggle/working/LoGoHTR')
from lightning.pytorch.cli import LightningCLI
from comer.datamodule import CROHMEDatamodule
from comer.lit_comer import LitCoMER

cli = LightningCLI(
    LitCoMER,
    CROHMEDatamodule,
    save_config_kwargs={"overwrite": True},
)
"""
with open('/kaggle/working/LoGoHTR/train.py', 'w') as f:
    f.write(new_train)
print("train.py fixed!")

In [ ]:
dataset_path = "/kaggle/input/SSL-HWD-Words(Labeled)"
print(dataset_path)

In [ ]:
!df -h /kaggle/working

In [ ]:
import zipfile
import pandas as pd
import io
import os
from PIL import Image

raw_path = '/kaggle/input/datasets/katyayanirk/logo-htr-data/SSL-HWD-Words(Labeled)'
out_path = '/kaggle/working/data.zip'

TRAIN_LIMIT = 5000
TEST_LIMIT = 1000
train_captions = []
test_captions = []
count = 0

writers = sorted(os.listdir(raw_path))
print(f"Total writers: {len(writers)}")

with zipfile.ZipFile(out_path, 'w', compression=zipfile.ZIP_DEFLATED) as dst_zip:
    for writer in writers:
        if count >= TRAIN_LIMIT + TEST_LIMIT:
            break
        writer_path = os.path.join(raw_path, writer)
        csv_file = os.path.join(writer_path, f'{writer}.csv')
        if not os.path.exists(csv_file):
            continue
        df = pd.read_csv(csv_file)
        for _, row in df.iterrows():
            if count >= TRAIN_LIMIT + TEST_LIMIT:
                break
            img_name = row['image_filename']
            word = str(row['transcription'])
            chars = ' '.join(list(word))
            fname = img_name.replace('.png', '')
            img_path = os.path.join(writer_path, img_name)
            if not os.path.exists(img_path):
                continue
            try:
                img = Image.open(img_path).convert('RGB')
                buf = io.BytesIO()
                img.save(buf, format='JPEG', quality=85)
                buf.seek(0)
                if count < TRAIN_LIMIT:
                    dst_zip.writestr(f'data/train/img/{fname}.jpg', buf.read())
                    train_captions.append(f'{fname} {chars}')
                else:
                    dst_zip.writestr(f'data/2014/img/{fname}.jpg', buf.read())
                    test_captions.append(f'{fname} {chars}')
                count += 1
                if count % 500 == 0:
                    print(f'Processed {count}/{TRAIN_LIMIT+TEST_LIMIT}')
            except:
                continue
    dst_zip.writestr('data/train/caption.txt', '\n'.join(train_captions))
    dst_zip.writestr('data/2014/caption.txt', '\n'.join(test_captions))

print(f"Done! {count} samples saved to data.zip")

In [ ]:
config = """
seed_everything: 7
trainer:
  callbacks:
    - class_path: lightning.pytorch.callbacks.LearningRateMonitor
      init_args:
        logging_interval: epoch
    - class_path: lightning.pytorch.callbacks.ModelCheckpoint
      init_args:
        save_top_k: 1
        monitor: val_WRR
        mode: max
        filename: '{epoch}-{step}-{val_WRR:.4f}'
        dirpath: /kaggle/working/checkpoints
  max_epochs: 10
  check_val_every_n_epoch: 2
  deterministic: true
  devices: 1
model:
  d_model: 256
  growth_rate: 24
  num_layers: 16
  nhead: 8
  num_decoder_layers: 3
  dim_feedforward: 1024
  dropout: 0.3
  dc: 32
  cross_coverage: true
  self_coverage: true
  beam_size: 10
  max_len: 200
  alpha: 1.0
  early_stopping: false
  temperature: 1.0
  learning_rate: 0.0008
  patience: 10
data:
  zipfile_path: /kaggle/working/data.zip
  test_year: "2014"
  train_batch_size: 8
  eval_batch_size: 4
  num_workers: 4
  scale_aug: true
"""
with open('/kaggle/working/LoGoHTR/config.yaml', 'w') as f:
    f.write(config)
print("Config updated!")

In [ ]:
%cd /kaggle/working/LoGoHTR
!python train.py fit --config config.yaml